In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Função de exportação
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))

from utils.export_utils import exportar_csv


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# GOLD 03 - QUAL É O CENÁRIO DE DIVERSIDADE DE GÊNERO NAS CARREIRAS DE DADOS?
# ---------------------------------------------------------------------

caminho_gold_03 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_03_diversidade_genero"

arquivos_gold_03 = [
    str(arquivo) for arquivo in caminho_gold_03.glob("part-*.csv")
]

if not arquivos_gold_03:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_03}"
    )

df_genero = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_03)
)


# ---------------------------------------------------------------------
# BASE DE SENIORIDADE
# ---------------------------------------------------------------------

"""
A composição geral é construída a partir da dimensão de senioridade para trabalhar com uma única contagem da população por edição e gênero. Isso evita somar, no mesmo indicador, registros provenientes de diferentes dimensões da Gold 03.
"""
df_nivel = (
    df_genero
    .filter(F.col("variavel") == "nivel")
    .groupBy("edicao", "genero", "valor")
    .agg(F.sum("contagem").alias("contagem"))
)


# ---------------------------------------------------------------------
# EVOLUÇÃO DA COMPOSIÇÃO GERAL POR GÊNERO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("1. EVOLUÇÃO DA COMPOSIÇÃO GERAL POR GÊNERO")
print("=" * 100)

"""
Os percentuais são calculados dentro de cada edição porque o tamanho das amostras varia entre os anos. No notebook, o total passa de 3.857 respondentes em 2023-2024 para 2.501 em 2025-2026, portanto a comparação histórica deve priorizar proporções em vez de contagens absolutas.
"""
composicao_genero = (
    df_nivel
    .groupBy("edicao", "genero")
    .agg(F.sum("contagem").alias("respondentes"))
)

janela_edicao = Window.partitionBy("edicao")

composicao_genero = (
    composicao_genero
    .withColumn(
        "total_edicao",
        F.sum("respondentes").over(janela_edicao)
    )
    .withColumn(
        "pct_geral",
        F.round(F.col("respondentes") / F.col("total_edicao") * 100, 1)
    )
    .orderBy("edicao", F.desc("respondentes"))
)

composicao_genero.show(100, truncate=False)
"""
A composição geral permanece majoritariamente masculina nas três edições. A participação feminina foi de 24,4% em 2023-2024, 24,7% em 2024-2025 e 22,7% em 2025-2026, enquanto a masculina chegou a 76,6% na edição mais recente.
"""


# ---------------------------------------------------------------------
# TOTAL DE RESPONDENTES POR EDIÇÃO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. TOTAL DE RESPONDENTES POR EDIÇÃO")
print("=" * 100)

total_por_edicao = (
    composicao_genero
    .select("edicao", "total_edicao")
    .distinct()
    .orderBy("edicao")
)

total_por_edicao.show(20, truncate=False)


# ---------------------------------------------------------------------
# REPRESENTATIVIDADE DE GÊNERO POR SENIORIDADE - HISTÓRICO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. REPRESENTATIVIDADE DE GÊNERO POR SENIORIDADE - HISTÓRICO")
print("=" * 100)

"""
A representatividade é calculada dentro de cada nível e edição. Assim, o percentual responde qual é a composição de gênero de Júnior, Pleno, Sênior e Especialista/Staff+, sem misturar diferenças de tamanho entre as senioridades.
"""
janela_nivel = Window.partitionBy("edicao", "valor")

genero_por_nivel = (
    df_nivel
    .withColumn(
        "total_nivel",
        F.sum("contagem").over(janela_nivel)
    )
    .withColumn(
        "pct_no_nivel",
        F.round(F.col("contagem") / F.col("total_nivel") * 100, 1)
    )
    .select(
        "edicao",
        F.col("valor").alias("nivel"),
        "genero",
        "contagem",
        "total_nivel",
        "pct_no_nivel"
    )
)


# Ordem dos níveis
genero_por_nivel = (
    genero_por_nivel
    .withColumn(
        "ordem_nivel",
        F.when(F.col("nivel") == "Júnior", 1)
        .when(F.col("nivel") == "Pleno", 2)
        .when(F.col("nivel") == "Sênior", 3)
        .when(F.col("nivel") == "Especialista/Staff", 4)
        .otherwise(99)
    )
    .orderBy("edicao", "ordem_nivel", F.desc("contagem"))
)

genero_por_nivel.select(
    "edicao", "nivel", "genero", "contagem", "total_nivel", "pct_no_nivel"
).show(100, truncate=False)


# ---------------------------------------------------------------------
# EVOLUÇÃO DA PARTICIPAÇÃO FEMININA POR SENIORIDADE
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. EVOLUÇÃO DA PARTICIPAÇÃO FEMININA POR SENIORIDADE")
print("=" * 100)

participacao_feminina_nivel = (
    genero_por_nivel
    .filter(F.col("genero") == "Feminino")
    .select(
        "edicao",
        "nivel",
        F.col("contagem").alias("mulheres"),
        "total_nivel",
        F.col("pct_no_nivel").alias("pct_feminino"),
        "ordem_nivel"
    )
    .orderBy("edicao", "ordem_nivel")
    .drop("ordem_nivel")
)

participacao_feminina_nivel.show(100, truncate=False)
"""
O histórico mostra participação feminina maior nos níveis iniciais do que nos níveis mais altos. Em 2025-2026, a participação é de 28,2% no Júnior, 22,4% no Pleno, 20,7% no Sênior e 20,1% em Especialista/Staff+.
"""


# ---------------------------------------------------------------------
# RETRATO ATUAL DA PARTICIPAÇÃO FEMININA POR SENIORIDADE
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. PARTICIPAÇÃO FEMININA POR SENIORIDADE | 2025-2026")
print("=" * 100)

participacao_feminina_atual = (
    participacao_feminina_nivel.filter(F.col("edicao") == "2025-2026")
)

participacao_feminina_atual.show(20, truncate=False)


# ---------------------------------------------------------------------
# BASE DE CARGOS
# ---------------------------------------------------------------------

"""
A análise por cargo utiliza apenas a dimensão cargo_atual para comparar a participação de gênero entre funções profissionais sem misturar outras dimensões da Gold 03.
"""
df_cargos = df_genero.filter(F.col("variavel") == "cargo_atual")


# Harmonização dos cargos de Engenharia e Arquitetura de Dados
"""
As nomenclaturas de Engenharia e Arquitetura de Dados são harmonizadas antes das comparações para que variações de rótulo não dividam o mesmo grupo profissional em categorias separadas.
"""
df_cargos = (
    df_cargos.withColumn(
        "cargo_harmonizado",
        F.when(
            F.col("valor").isin(
                "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
                "Engenheiro de Dados/Data Engineer/Data Architect",
                "Arquiteto de Dados/Data Architect"
            ),
            "Engenharia e Arquitetura de Dados"
        ).otherwise(F.col("valor"))
    )
)


# ---------------------------------------------------------------------
# REPRESENTATIVIDADE DE GÊNERO POR CARGO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. REPRESENTATIVIDADE DE GÊNERO POR CARGO")
print("=" * 100)

df_cargos_consolidado = (
    df_cargos
    .groupBy("edicao", "cargo_harmonizado", "genero")
    .agg(F.sum("contagem").alias("contagem"))
)

janela_cargo = Window.partitionBy("edicao", "cargo_harmonizado")

genero_por_cargo = (
    df_cargos_consolidado
    .withColumn(
        "total_cargo",
        F.sum("contagem").over(janela_cargo)
    )
    .withColumn(
        "pct_no_cargo",
        F.round(F.col("contagem") / F.col("total_cargo") * 100, 1)
    )
    .orderBy(
        "edicao",
        F.desc("total_cargo"),
        F.desc("contagem")
    )
)

genero_por_cargo.show(200, truncate=False)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR CARGO - EDIÇÃO ATUAL
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. PARTICIPAÇÃO FEMININA POR CARGO | 2025-2026")
print("=" * 100)

participacao_feminina_cargo_atual = (
    genero_por_cargo
    .filter(
        (F.col("edicao") == "2025-2026")
        & (F.col("genero") == "Feminino")
        & (F.col("total_cargo") >= 20)
        & (F.col("cargo_harmonizado") != "Outra Opção")
    )
    .select(
        "cargo_harmonizado",
        F.col("contagem").alias("mulheres"),
        "total_cargo",
        F.col("pct_no_cargo").alias("pct_feminino")
    )
    .orderBy(
        F.desc("pct_feminino"),
        F.desc("total_cargo")
    )
)

participacao_feminina_cargo_atual.show(100, truncate=False)
"""
Na edição 2025-2026, entre os cargos com pelo menos 20 respondentes, Data Product Manager/Product Manager apresenta a maior participação feminina, com 42,4%. Entre os menores percentuais aparecem Engenharia e Arquitetura de Dados com 17,7%, Machine Learning/AI Engineer com 15,1% e Outras Engenharias com 3,8%.
"""


# ---------------------------------------------------------------------
# COMPARABILIDADE HISTÓRICA DOS CARGOS
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. COMPARABILIDADE HISTÓRICA DOS CARGOS")
print("=" * 100)

"""
Antes da análise histórica por cargo, é verificado se cada função aparece nas três edições e qual foi sua menor amostra. Essa checagem evita interpretar como tendência uma diferença baseada em categorias pouco representadas ou sem continuidade no período.
"""
amostra_cargo_edicao = (
    genero_por_cargo
    .select("edicao", "cargo_harmonizado", "total_cargo")
    .distinct()
)

comparabilidade_cargos = (
    amostra_cargo_edicao
    .groupBy("cargo_harmonizado")
    .agg(
        F.countDistinct("edicao").alias("qtd_edicoes"),
        F.min("total_cargo").alias("menor_amostra"),
        F.max("total_cargo").alias("maior_amostra")
    )
    .orderBy(
        F.desc("qtd_edicoes"),
        F.desc("menor_amostra")
    )
)

comparabilidade_cargos.show(100, truncate=False)


# ---------------------------------------------------------------------
# EVOLUÇÃO DA PARTICIPAÇÃO FEMININA POR CARGO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("9. EVOLUÇÃO DA PARTICIPAÇÃO FEMININA POR CARGO")
print("=" * 100)

"""
Para o histórico são mantidos somente cargos presentes nas três edições, com pelo menos 20 respondentes em cada uma e com perfil profissional identificável. No notebook, categorias como Professor/Pesquisador e Estatístico não atingem o corte mínimo, enquanto categorias presentes em apenas uma edição também não avançam para a comparação.
"""
cargos_elegiveis = (
    comparabilidade_cargos.filter(
        (F.col("qtd_edicoes") == 3)
        & (F.col("menor_amostra") >= 20)
        & (F.col("cargo_harmonizado") != "Outra Opção")
    )
)

participacao_feminina_cargo_historica = (
    genero_por_cargo
    .filter(F.col("genero") == "Feminino")
    .join(
        cargos_elegiveis.select("cargo_harmonizado"),
        on="cargo_harmonizado",
        how="inner"
    )
    .select(
        "edicao",
        "cargo_harmonizado",
        F.col("contagem").alias("mulheres"),
        "total_cargo",
        F.col("pct_no_cargo").alias("pct_feminino")
    )
    .orderBy("cargo_harmonizado", "edicao")
)

participacao_feminina_cargo_historica.show(200, truncate=False)
"""
A evolução por cargo não é uniforme. Data Product Manager/Product Manager permanece acima de 38% de participação feminina nas três edições, enquanto outras funções apresentam oscilações ou redução, como Analista de Dados, que passa de 28,6% em 2023-2024 para 24,5% em 2025-2026.
"""


# ---------------------------------------------------------------------
# BASE DE FAIXA SALARIAL
# ---------------------------------------------------------------------

"""
A análise salarial é feita a partir da distribuição de gênero dentro de cada faixa de remuneração. O denominador é recalculado por edição e faixa para que a participação feminina seja comparável entre intervalos com tamanhos diferentes.
"""
df_salario = (
    df_genero
    .filter(F.col("variavel") == "faixa_salarial")
    .groupBy("edicao", "genero", "valor")
    .agg(F.sum("contagem").alias("contagem"))
)


# ---------------------------------------------------------------------
# TOTAL DE RESPONDENTES POR FAIXA SALARIAL
# ---------------------------------------------------------------------

janela_faixa = Window.partitionBy("edicao", "valor")

df_salario = (
    df_salario.withColumn(
        "total_faixa",
        F.sum("contagem").over(janela_faixa)
    )
)


# ---------------------------------------------------------------------
# FAIXAS SALARIAIS DESCARTADAS POR BAIXA AMOSTRA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("10. FAIXAS SALARIAIS DESCARTADAS - AMOSTRA <= 1")
print("=" * 100)

faixas_salariais_descartadas = (
    df_salario
    .select(
        "edicao",
        F.col("valor").alias("faixa_salarial"),
        "total_faixa"
    )
    .distinct()
    .filter(F.col("total_faixa") <= 1)
    .orderBy("edicao", "faixa_salarial")
)

faixas_salariais_descartadas.show(100, truncate=False)
"""
A inspeção confirma que somente duas faixas possuem amostra igual a um respondente. A remoção é aplicada de forma objetiva pelo tamanho da amostra, sem excluir as demais faixas salariais válidas.
"""


# Remover faixas com apenas 1 respondente
"""
As faixas com apenas um respondente são retiradas das análises posteriores porque um único caso produziria percentuais sem representatividade. O notebook identificou duas situações: "de R$ 101/mês a R$ 2.000/mês" em 2023-2024 e "de R$ 25.001/mês a R$ 3000/mês" em 2025-2026.
"""
df_salario_valido = df_salario.filter(F.col("total_faixa") > 1)


# ---------------------------------------------------------------------
# REPRESENTATIVIDADE DE GÊNERO POR FAIXA SALARIAL
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("11. REPRESENTATIVIDADE DE GÊNERO POR FAIXA SALARIAL")
print("=" * 100)

genero_por_faixa_salarial = (
    df_salario_valido
    .withColumn(
        "pct_na_faixa",
        F.round(F.col("contagem") / F.col("total_faixa") * 100, 1)
    )
    .select(
        "edicao",
        F.col("valor").alias("faixa_salarial"),
        "genero",
        "contagem",
        "total_faixa",
        "pct_na_faixa"
    )
)


# Ordem das faixas salariais
"""
As faixas salariais recebem uma ordem numérica apenas para preservar a sequência econômica dos intervalos nas tabelas e análises históricas. O percentual continua sendo calculado diretamente sobre a contagem observada em cada faixa.
"""
genero_por_faixa_salarial = (
    genero_por_faixa_salarial
    .withColumn(
        "ordem_faixa",
        F.when(F.col("faixa_salarial") == "Menos de R$ 1.000/mês", 1)
        .when(F.col("faixa_salarial") == "de R$ 1.001/mês a R$ 2.000/mês", 2)
        .when(F.col("faixa_salarial") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
        .when(F.col("faixa_salarial") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
        .when(F.col("faixa_salarial") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
        .when(F.col("faixa_salarial") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
        .when(F.col("faixa_salarial") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
        .when(F.col("faixa_salarial") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
        .when(F.col("faixa_salarial") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
        .when(F.col("faixa_salarial") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
        .when(F.col("faixa_salarial") == "de R$ 25.001/mês a R$ 30.000/mês", 11)
        .when(F.col("faixa_salarial") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
        .when(F.col("faixa_salarial") == "Acima de R$ 40.001/mês", 13)
        .otherwise(99)
    )
    .orderBy("edicao", "ordem_faixa", F.desc("contagem"))
)

genero_por_faixa_salarial.select(
    "edicao",
    "faixa_salarial",
    "genero",
    "contagem",
    "total_faixa",
    "pct_na_faixa"
).show(300, truncate=False)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR FAIXA SALARIAL - HISTÓRICO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("12. EVOLUÇÃO DA PARTICIPAÇÃO FEMININA POR FAIXA SALARIAL")
print("=" * 100)

participacao_feminina_salario_historica = (
    genero_por_faixa_salarial
    .filter(F.col("genero") == "Feminino")
    .select(
        "edicao",
        "faixa_salarial",
        F.col("contagem").alias("mulheres"),
        "total_faixa",
        F.col("pct_na_faixa").alias("pct_feminino"),
        "ordem_faixa"
    )
    .orderBy("edicao", "ordem_faixa")
    .drop("ordem_faixa")
)

participacao_feminina_salario_historica.show(200, truncate=False)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR FAIXA SALARIAL - EDIÇÃO ATUAL
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("13. PARTICIPAÇÃO FEMININA POR FAIXA SALARIAL | 2025-2026")
print("=" * 100)

participacao_feminina_salario_atual = (
    genero_por_faixa_salarial
    .filter(
        (F.col("edicao") == "2025-2026")
        & (F.col("genero") == "Feminino")
    )
    .select(
        "faixa_salarial",
        F.col("contagem").alias("mulheres"),
        "total_faixa",
        F.col("pct_na_faixa").alias("pct_feminino"),
        "ordem_faixa"
    )
    .orderBy("ordem_faixa")
    .drop("ordem_faixa")
)

participacao_feminina_salario_atual.show(100, truncate=False)
"""
Na edição 2025-2026, a participação feminina tende a ser menor nas faixas salariais mais altas: é de 31,5% entre R$ 2.001 e R$ 3.000, 18,3% entre R$ 20.001 e R$ 25.000 e 7,0% acima de R$ 40.001. A leitura é feita por faixa, sem assumir uma queda perfeitamente linear entre todos os intervalos.
"""


# ---------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS
# ---------------------------------------------------------------------

exportar_csv(
    composicao_genero,
    OUTPUT_DIR,
    "composicao_genero_historica.csv"
)

exportar_csv(
    participacao_feminina_nivel,
    OUTPUT_DIR,
    "participacao_feminina_senioridade_historica.csv"
)

exportar_csv(
    participacao_feminina_cargo_historica,
    OUTPUT_DIR,
    "participacao_feminina_cargo_historica.csv"
)

exportar_csv(
    participacao_feminina_salario_historica,
    OUTPUT_DIR,
    "participacao_feminina_salario_historica.csv"
)